# 04 - Direction B: rewrite the customer message with an LLM before searching

**Idea.** One Gemini call turns the raw message into `{is_kb_question, intent, search_queries}`:
- `is_kb_question` is a **router**: False for messages that should never be answered from the KB (policy requests, prompt injection,
  social engineering, outage triage...).
- `intent` is a one-sentence restatement; `search_queries` are up to 3 short keyword queries.
Retrieval then searches each sub-query (hybrid, top 20), merges them with RRF and reranks against either the **intent**
(`B_rewrite_intent`) or the **original message** (`B_rewrite_orig`).

**What is measured.** Retrieval quality vs the raw pipeline, router accuracy, the same gate sweeps as notebook 03 applied to B's scores
(and "gate = router only" / "router AND score"), and the price: LLM calls, tokens and latency per query.

| | |
|---|---|
| **Required caches** | `cache/rewrites.json` (`uv run --env-file .env python -m experiments.llm.rewrite`, 89 Gemini calls) and then `cache/runs/B_rewrite_intent.json`, `B_rewrite_orig.json` (re-run `uv run python -m experiments.run_variants` AFTER the rewrites exist), plus `R3_current.json`, `A_clean.json`. |
| **Needed for real labels** | `cache/judgments.json` (`uv run python -m experiments.build_pools` then `uv run --env-file .env python -m experiments.llm.judge`). Without it gate sweeps use a provisional label (ANS / OOD+ADJ) and say so; retrieval metrics are skipped. |

In [ ]:
import sys, pathlib, math, warnings
warnings.filterwarnings("ignore")
_here = pathlib.Path.cwd().resolve()
REPO = next(p for p in [_here, *_here.parents] if (p / "experiments" / "lab").is_dir())
sys.path[:0] = [str(REPO), str(REPO / "experiments")]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lab import notebook_utils as nu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})
print("cache directory:", nu.cache_dir())

In [ ]:
RW = nu.load_rewrites()
RUNS = nu.load_runs(["R3_current", "A_clean", "R0_lex", "R1_dense", "B_rewrite_intent", "B_rewrite_orig"])
HAVE_RW = RW is not None
HAVE_B = HAVE_RW and "B_rewrite_intent" in RUNS and "B_rewrite_orig" in RUNS
nu.need(HAVE_RW, "rewrites.json", nu.CMD_REWRITE)
if HAVE_RW:
    nu.need(HAVE_B, "runs B_rewrite_intent / B_rewrite_orig", nu.CMD_RUN_VARIANTS + "   (after the rewrites exist)")
Q = nu.load_queries()
J = nu.load_judgments()
HAVE_J = J is not None
QR = nu.qrels_from(J)
labels = nu.gold_labels(Q, J)
should = labels["label"].dropna().astype(bool)
print(nu.label_banner(labels))
if HAVE_RW:
    print(f"rewrites cached for {len(RW)} of {len(Q)} queries")

## 1. Ten example rewrites, side by side with the raw text

In [ ]:
if nu.need(HAVE_RW, "rewrites.json", nu.CMD_REWRITE):
    rng = np.random.default_rng(3)
    pick = []
    for s, k in (("TKT", 4), ("SCN", 2), ("ANS", 1), ("OOD", 1), ("ADJ", 1)):
        ids = [i for i in Q.index[Q["set"] == s] if i in RW]
        pick += list(rng.choice(ids, size=min(k, len(ids)), replace=False)) if ids else []
    extra = [i for i in Q.index if nu.never_kb(i) and i in RW and i not in pick]
    pick = (pick + extra)[:10]
    ex = pd.DataFrame([{"id": i, "set": Q.loc[i, "set"], "raw message": Q.loc[i, "text"][:160], "is_kb_question": RW[i]["is_kb_question"],
                        "intent": RW[i]["intent"], "search_queries": " | ".join(RW[i]["search_queries"])} for i in pick]).set_index("id")
    pd.set_option("display.max_colwidth", 160)
    display(ex)

## 2. Router accuracy

Messages that must **never** hit the KB: SC-03 (SLA credit), SC-04 (MFA social engineering), SC-05 (prompt injection),
SC-07 (multi-site outage / tier claim), SC-10 (C2 whitelist request), TCK-20264223, -246, -209, -210. The router is correct when it says
`is_kb_question = False` for them and `True` for everything else that is a genuine question. (OOD/ADJ are genuine questions the KB cannot answer, so `True` is the
expected router output there: refusing them is the gate's job, not the router's.)

In [ ]:
if nu.need(HAVE_RW, "rewrites.json", nu.CMD_REWRITE):
    R = pd.DataFrame({"id": Q.index, "set": Q["set"], "never_kb": [nu.never_kb(i) for i in Q.index],
                      "is_kb": [RW[i]["is_kb_question"] if i in RW else None for i in Q.index]}).set_index("id").dropna(subset=["is_kb"])
    R["is_kb"] = R["is_kb"].astype(bool)
    R["correct"] = R["is_kb"] == ~R["never_kb"]
    nk = R[R["never_kb"]]; ok = R[~R["never_kb"]]
    router = pd.DataFrame({"n": [len(nk), len(ok), len(R)],
                           "router correct": [int(nk["correct"].sum()), int(ok["correct"].sum()), int(R["correct"].sum())]},
                          index=["never-KB messages (should be is_kb=False)", "all other messages (should be is_kb=True)", "all"])
    router["accuracy"] = router["router correct"] / router["n"]
    display(router)
    print("never-KB messages the router let through (would be answered from the KB):", list(nk.index[nk["is_kb"]]))
    print("normal messages the router wrongly blocked:", list(ok.index[~ok["is_kb"]]))
    display(R.groupby("set")["is_kb"].mean().reindex(nu.SETS).to_frame("share with is_kb=True"))

## 3. Retrieval quality: raw vs cleaned vs rewrite

In [ ]:
NAMES = [n for n in ("R3_current", "A_clean", "B_rewrite_intent", "B_rewrite_orig") if n in RUNS]
if nu.need(HAVE_B and HAVE_J, "judgments.json and B runs", nu.CMD_POOLS + " ; " + nu.CMD_JUDGE):
    PQ = {n: nu.per_query_metrics(RUNS[n], QR) for n in NAMES}
    RET = pd.DataFrame({n: m.mean() for n, m in PQ.items()}).T
    display(RET)
    rows = []
    for b in ("B_rewrite_intent", "B_rewrite_orig"):
        for metric in ("hit@1", "hit@5", "ndcg@5"):
            x, y = PQ[b][metric], PQ["R3_current"][metric]; ok_ = x.notna() & y.notna()
            d = nu.bootstrap_diff(nu.mean_stat(x[ok_].to_numpy()), nu.mean_stat(y[ok_].to_numpy()), int(ok_.sum()))
            rows.append({"variant": b + " - R3_current", "metric": metric, "diff": d["diff"], "CI low": d["lo"], "CI high": d["hi"], "zero inside CI": d["lo"] <= 0 <= d["hi"]})
    display(pd.DataFrame(rows).set_index("variant"))

## 4. Gate sweeps applied to B

Same gates as notebook 03 (`top1`, `margin`, `z`) computed on B's rerank scores, plus two that use the router:
`router_only` (answer iff `is_kb_question`) and `top1_routed` (answer iff the router says KB **and** top-1 >= t).

In [ ]:
GATES_B = ["top1", "margin", "z", "router_only", "top1_routed"]
FB, STUDY_B, SWEEPS_B = {}, {}, {}
if nu.need(HAVE_B and len(should) > 0, "rewrites + B runs", nu.CMD_REWRITE):
    for n in ("B_rewrite_intent", "B_rewrite_orig"):
        f = nu.gate_features(RUNS[n])
        kb = pd.Series({i: bool(RW[i]["is_kb_question"]) if i in RW else True for i in f.index})
        f["router_only"] = kb.astype(float)
        f["top1_routed"] = f["top1"].where(kb, -math.inf)
        FB[n] = f
        STUDY_B[n], SWEEPS_B[n] = nu.gate_study(f, should, GATES_B)
    display(pd.concat(STUDY_B, names=["run"]))
    fig = nu.plot_sweeps(SWEEPS_B["B_rewrite_intent"], "B_rewrite_intent: F1 / false answers / false refusals vs threshold"); plt.show()

## 5. What B costs: LLM calls, tokens, latency per query

`latency` = the rewrite call plus retrieval (1-3 sub-query searches and a rerank). The retrieval time comes from the run cache, the LLM time from
`rewrites.json`. The raw pipeline has no LLM call.

In [ ]:
if nu.need(HAVE_B, "rewrites + B runs", nu.CMD_REWRITE):
    C = nu.rewrite_costs(RW)
    tot_ms = {n: C["latency_ms"].reindex(Q.index) + pd.Series({i: RUNS[n][i]["latency_ms"] for i in Q.index if i in RUNS[n]}) for n in ("B_rewrite_intent", "B_rewrite_orig")}
    cost = pd.DataFrame({
        "LLM calls / query": [0, C["requests"].mean(), C["requests"].mean()],
        "tokens / query (in+out)": [0, (C["in_tokens"] + C["out_tokens"]).mean(), (C["in_tokens"] + C["out_tokens"]).mean()],
        "p50 latency ms": [nu.p50([v["latency_ms"] for v in RUNS["R3_current"].values()]), nu.p50(tot_ms["B_rewrite_intent"]), nu.p50(tot_ms["B_rewrite_orig"])],
        "p50 of the LLM part ms": [0, nu.p50(C["latency_ms"]), nu.p50(C["latency_ms"])],
    }, index=["R3_current (raw)", "B_rewrite_intent", "B_rewrite_orig"])
    display(cost)

# RESULT TABLE - Direction B (best gate per variant, next to the raw pipeline)

`F1` is the answer/refuse F1 at the best (in-sample) threshold of the best gate; retrieval columns need judgments.

In [ ]:
if nu.need(HAVE_B and bool(STUDY_B), "rewrites + B runs", nu.CMD_REWRITE):
    llm = float(C["requests"].mean()); tok = float((C["in_tokens"] + C["out_tokens"]).mean())
    p50_by_run = {"R3_current": cost.loc["R3_current (raw)", "p50 latency ms"], "B_rewrite_intent": cost.loc["B_rewrite_intent", "p50 latency ms"],
                  "B_rewrite_orig": cost.loc["B_rewrite_orig", "p50 latency ms"]}
    rows = []
    def add(name, feats, gates, is_llm):
        bc = nu.best_config(feats, should, gates)
        if not bc:
            return
        ci = nu.bootstrap_ci(nu.f1_stat(should.tolist(), bc["did"].tolist()), len(should))
        run = bc["run"]
        r = {"approach": name, "best gate": f"{bc['gate']} >= {bc['threshold']:.2f}", "answer F1": bc["f1"], "F1 95% CI": f"[{ci[1]:.2f}, {ci[2]:.2f}]",
             "false-answer rate": bc["false_answer_rate"], "false-refusal rate": bc["false_refusal_rate"]}
        if HAVE_J and run in PQ:
            r["hit@1"], r["hit@5"] = PQ[run]["hit@1"].mean(), PQ[run]["hit@5"].mean()
        r["p50 latency ms"] = p50_by_run[run]; r["LLM calls/query"] = llm if is_llm else 0.0; r["extra tokens/query"] = tok if is_llm else 0.0
        rows.append(r)
    add("R3_current (raw)", {"R3_current": nu.gate_features(RUNS["R3_current"])}, ["top1", "margin", "z"], False)
    for n in ("B_rewrite_intent", "B_rewrite_orig"):
        add(n, {n: FB[n]}, GATES_B, True)
    table = pd.DataFrame(rows).set_index("approach")
    display(table)
    print(nu.label_banner(labels))
    print(f"Router accuracy overall: {router.loc['all', 'accuracy']:.0%}; never-KB messages correctly blocked: {router.iloc[0]['router correct']:.0f} of {router.iloc[0]['n']:.0f}.")